In [4]:
import sys
import os

sys.path.append(
    os.path.abspath(".")
)

In [5]:
sys.path.append(
    os.path.abspath("..")
)

In [6]:
from src.datasets.brats_dataset import create_file_list
from src.config import DATA_DIR
from src.transforms.preprocessing import (
    train_transform,
    val_transform
)

In [7]:
files = create_file_list(DATA_DIR)

print(len(files))

369


In [8]:
from sklearn.model_selection import train_test_split

files = create_file_list(DATA_DIR)

# 70% train
train_files, temp_files = train_test_split(
    files,
    test_size=0.30,
    random_state=42,
    shuffle=True
)

# 15% val, 15% test
val_files, test_files = train_test_split(
    temp_files,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

print("Train:", len(train_files))
print("Val  :", len(val_files))
print("Test :", len(test_files))

Train: 258
Val  : 55
Test : 56


In [9]:
from monai.data import Dataset

train_ds = Dataset(
    data=train_files,
    transform=train_transform
)

val_ds = Dataset(
    data=val_files,
    transform=val_transform
)

test_ds = Dataset(
    data=test_files,
    transform=val_transform
)

In [10]:
from src.graphs.create_graph_dataset import create_graph_dataset

In [11]:
train_graphs = create_graph_dataset(train_ds)

val_graphs = create_graph_dataset(val_ds)

test_graphs = create_graph_dataset(test_ds)

  0%|          | 0/258 [00:00<?, ?it/s]d:\Internship\brats2020\src\graphs\graph_dataset.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  edge_index = torch.tensor(
  0%|          | 1/258 [00:05<21:28,  5.01s/it]d:\Internship\brats2020\src\graphs\graph_dataset.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  edge_index = torch.tensor(
  1%|          | 2/258 [00:07<14:54,  3.49s/it]d:\Internship\brats2020\src\graphs\graph_dataset.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  edge_index = torch.tensor(
  1%|          | 3/258 [

In [12]:
from torch_geometric.loader import DataLoader

train_loader = DataLoader(
    train_graphs,
    batch_size=1,
    shuffle=True
)

val_loader = DataLoader(
    val_graphs,
    batch_size=1,
    shuffle=False
)

test_loader = DataLoader(
    test_graphs,
    batch_size=1,
    shuffle=False
)

In [13]:
from train.train_graphsage import train_graphsage

model = train_graphsage(
    train_loader,
    val_loader,
    epochs=300,
    lr=1e-3,
    hidden=128
)


Training Class Distribution:
Counter({np.int64(0): 812230, np.int64(2): 41524, np.int64(3): 8981, np.int64(1): 8015})

Class Weights:
tensor([ 0.2680, 27.1600,  5.2425, 24.2387])
Epoch 000 | Train Loss: 0.8241 | Train Acc: 0.7424 | Val Loss: 0.4627 | Val Acc: 0.9462
Epoch 010 | Train Loss: 0.5776 | Train Acc: 0.8476 | Val Loss: 0.4183 | Val Acc: 0.9436
Epoch 020 | Train Loss: 0.5655 | Train Acc: 0.8623 | Val Loss: 0.4201 | Val Acc: 0.9485
Epoch 030 | Train Loss: 0.5509 | Train Acc: 0.8753 | Val Loss: 0.4085 | Val Acc: 0.9770
Epoch 040 | Train Loss: 0.5446 | Train Acc: 0.8838 | Val Loss: 0.4049 | Val Acc: 0.9613
Epoch 050 | Train Loss: 0.5402 | Train Acc: 0.8824 | Val Loss: 0.3997 | Val Acc: 0.9674
Epoch 060 | Train Loss: 0.5271 | Train Acc: 0.8905 | Val Loss: 0.4027 | Val Acc: 0.9638

Early stopping at epoch 60

Training Complete
Best Validation Accuracy: 0.9770


In [14]:
from evaluate_graphsage import evaluate_graphsage

evaluate_graphsage(
    model,
    test_loader
)


===== Evaluation Results =====
Accuracy  : 0.976257900995056
Precision : 0.6335499756501
Recall    : 0.7140639113382586
F1 Score  : 0.6525363571665407

Confusion Matrix:

[[859187    119  11786   1867]
 [   460   1639   1379    243]
 [  3389    655  10591    567]
 [   314     96    370   2162]]

Classification Report:

              precision    recall  f1-score   support

           0       1.00      0.98      0.99    872959
           1       0.65      0.44      0.53      3721
           2       0.44      0.70      0.54     15202
           3       0.45      0.73      0.56      2942

    accuracy                           0.98    894824
   macro avg       0.63      0.71      0.65    894824
weighted avg       0.98      0.98      0.98    894824


===== Dice Scores =====
Class 0: 0.9897
Class 1: 0.5262
Class 2: 0.5386
Class 3: 0.5557

===== BraTS Region Dice Scores =====
WT (Whole Tumor)      : 0.6638
TC (Tumor Core)       : 0.5910
ET (Enhancing Tumor)  : 0.5557
